# 03. Архитектура, posterior и API кодека

<!-- study-navigation:start -->
**Основа:** [MIT 6.S184 / 2026](https://diffusion.csail.mit.edu/2026/index.html) · [Основной конспект](https://diffusion.csail.mit.edu/2026/docs/lecture_notes.pdf) · [Учебная карта](../../curriculum/README.md) · [Полное содержание](../../curriculum/CONTENTS.md).

**Связанные лекции:** [Lecture 4](https://diffusion.csail.mit.edu/2026/docs/20260128_Lecture_04_edited.pdf).
<!-- study-navigation:end -->

Сопоставим все классы с вычислительным графом, проследим BCTHW hooks и проверим реальный return contract encode.

[Порядок и источники](../README.md) · [Связь с MIT Lab 3](../MIT_LAB3_BRIDGE.md) · [Все компоненты и методы](../SOURCE_MAP.md) · [Результаты](../reports/VALIDATION.md).

Работаем по циклу **формула → прогноз → код → измерение → вывод**. Все ячейки по умолчанию используют CPU; никакие веса, датасеты или ComfyUI jobs не скачиваются/запускаются. Исходные алгоритмы и учебные адаптеры явно различаются. Перед запуском численного опыта сформулируйте ожидаемый результат.


**MIT мост:** Q4.2 AttentionBlock (58–59), Q4.4 Encoder (62–63), Q4.6 Decoder (66–67), Q4.7 VAE (68–69), затем Q5.1 LatentCFGTrainer (76–77). Сравните tuple(mean,logvar) из MIT с distribution API промышленной VAE. [Контракты](../MIT_LAB3_BRIDGE.md).


In [ ]:
from pathlib import Path
import sys, json, math
import numpy as np
import torch
from torch import nn
import torch.nn.functional as F
import matplotlib.pyplot as plt
candidates=[Path.cwd(),*Path.cwd().parents]
COURSE=next((p/'vae' for p in candidates if (p/'vae/labkit').is_dir()),None)
assert COURSE is not None, 'Clone the repository first; see the Colab instructions in the course README. No weights are needed.'
sys.path.insert(0,str(COURSE))
from labkit.vae_lab import (NATIVE,TinyCodec,DiagonalPosterior,oracle_codec,plan,
    axis_coverage,anchored_starts,synthetic_video,errors,measure,show_source,
    source_hash,save_metrics,figure_path)
torch.set_num_threads(2)
torch.manual_seed(42)
LAB='03_architecture_posterior_and_api';metrics={}
print('Pinned source:',source_hash())


## 1. Полный путь encoder и decoder

Encoder: causal input conv → четыре DownBlocks → MidBlock → GroupNorm/SiLU/output conv → quant_conv. Ширины released defaults: 128,256,512,512; residual blocks по 2 на stage. Spatial downsample у первых трёх stages; temporal — у второго и третьего.

| После stage | C defaults | F | H/W |
|---|---|---|---|
| Input / conv_in | 3 → 128 | F | H,W |
| Down 0 | 128 | F | H/2,W/2 |
| Down 1 | 256 | ceil(F/2) | H/4,W/4 |
| Down 2 | 512 | ceil(F/4) | H/8,W/8 |
| Down 3 / mid | 512 | F_z | H/8,W/8 |
| conv_out / quant_conv | 32 | F_z | H/8,W/8 |
| posterior mode/sample | 16 | F_z | H/8,W/8 |

`quant_conv` — learned 1×1×1 projection posterior moments, **не discrete quantizer и не VQ codebook**. `post_quant_conv` — learned projection перед decoder. Decoder идёт по reversed widths, использует layers_per_block+1 residuals, spatial upsampling на stages 0–2, temporal на 1–2. Optional MidBlock attention получает flattened FHW tokens и causal frame mask; quadratic attention и dense mask могут доминировать память.

Для измерений уменьшим widths до 4,8,8,8 и layers до 1, сохранив четыре stages и геометрию. Mid attention выключена; это явно ограничивает перенос результатов.


In [ ]:
codec=TinyCodec(seed=3);x=synthetic_video(9,32,32)
shapes=[];handles=[]
for side,net,blocks in [('enc',codec.encoder,codec.encoder.down_blocks),('dec',codec.decoder,codec.decoder.up_blocks)]:
    for i,block in enumerate(blocks):
        handles.append(block.register_forward_hook(lambda m,a,b,label=f'{side}.{i}':shapes.append((label,list(b.shape)))))
with torch.inference_mode():
    moments=codec._encode(x);posterior=DiagonalPosterior(moments)
    rec=codec._decode(posterior.mode()).sample
for h in handles:h.remove()
assert moments.shape==(1,4,3,4,4) and rec.shape==x.shape
metrics['stage_shapes']=shapes;metrics['tiny_parameters']=sum(p.numel() for p in codec.parameters())
print('native tiny widths: 4,8,8,8; latent channels: 2')
for row in shapes:print(row)


## 1b. Mid attention: из feature volume в Q/K/V

При включённой attention X имеет B,C,F,H,W. Permute/flatten даёт B,N,C, N=FHW. Для одного head:

$$
Q=XW_Q,\quad K=XW_K,\quad V=XW_V,\qquad
Y=\mathrm{softmax}(QK^\top/\sqrt d+M)V.
$$

Форма Q/K/V — B,h,N,d, где C=h·d. Mask M равен 0 для key frame≤query frame и −∞ для будущего; spatial positions внутри одного кадра видят друг друга. В source encoder/decoder передают attention_head_dim равным ширине mid, то есть при C=512 получается один head с d=512; не переносите multihead настройки DiT на VAE автоматически.

Полный score/mask tensor растёт как N². Tiling уменьшает N внутри одного вызова, но одновременно убирает взаимодействия за пределами tile. Ниже вычислим attention по формуле на маленьком тензоре и сравним с PyTorch SDPA. Это **учебная реализация attention formula**, не импорт Diffusers Attention и не numerical parity с pretrained mid block.


In [ ]:
q=torch.randn(1,1,6,4);k=torch.randn_like(q);v=torch.randn_like(q)
mask=NATIVE.prepare_causal_attention_mask(3,2,q.dtype,q.device,1).unsqueeze(1)
manual=torch.softmax(q@k.transpose(-1,-2)/math.sqrt(4)+mask,dim=-1)@v
sdpa=F.scaled_dot_product_attention(q,k,v,attn_mask=mask)
assert torch.allclose(manual,sdpa,atol=2e-6)
metrics['attention_formula_sdpa_max_error']=float((manual-sdpa).abs().max())
show_source('HunyuanVideoMidBlock3D')


## 2. Return contract: encode не возвращает готовый tensor

В **этом** snapshot `encode(..., return_dict=True)` возвращает `AutoencoderKLOutput(latent_dist=posterior)`, а `return_dict=False` — tuple с posterior. Итак:

- `vae.encode(x)[0]` — distribution object, не latent tensor;
- deterministic latent: `vae.encode(x).latent_dist.mode()`;
- stochastic latent: `.latent_dist.sample(generator=...)`.

В K6 wrapper `prepare_latents.encode_video` ожидается tensor после `[0]`. Это отдельная несовместимость контрактов закреплённой пары caller/callee; она не доказывает, что любой runtime K6 broken (может использовать другую VAE/adapter). Старый tuple[tensor] double проверяет ожидаемый wrapper interface, а не настоящий VAE API.

Ниже body `encode` оригинальный, posterior/output dependency — учебный adapter с тем же смыслом. Актуальный Diffusers не установлен в root env; не выдаём adapter за его runtime.


In [ ]:
show_source('encode')
# opt_tiling=False avoids overriding explicit full-shape test configuration.
with torch.inference_mode():
    output=codec.encode(x,opt_tiling=False)
    wrong=output[0]
    assert isinstance(wrong,DiagonalPosterior) and not torch.is_tensor(wrong)
    mode=output.latent_dist.mode()
    sample=output.latent_dist.sample(generator=torch.Generator().manual_seed(9))
metrics['posterior_contract']={'getitem_is_tensor':torch.is_tensor(wrong),'mode_shape':list(mode.shape),
 'mode_sample_MSE':float((mode-sample).square().mean())}
print(metrics['posterior_contract'])


## 3. Scaling: смена единиц перед DiT

Пусть s=0.476986 из config. В generative pipeline z_DiT=s·z_VAE; перед decoder делим обратно:

$$
z_{\mathrm{DiT}}=s z,\qquad z=z_{\mathrm{DiT}}/s,\qquad
\mathrm{Cov}(s z)=s^2\mathrm{Cov}(z).
$$

Scaling не делает распределение автоматически N(0,I), не заменяет KL и не входит сам собой в `decode`. В учебном вызове `_decode` подаём unscaled z. Проверим roundtrip единиц и ошибку двойного масштабирования; outputs случайной сети не оценивают perceptual quality.


In [ ]:
s=codec.config.scaling_factor
with torch.inference_mode():
    correct=codec._decode((s*mode)/s).sample
    wrong_scale=codec._decode(s*mode).sample
assert torch.allclose(correct,rec,atol=2e-6)
metrics['correct_scale_roundtrip']=errors(rec,correct)
metrics['missing_inverse_scale']=errors(rec,wrong_scale)
print(metrics['missing_inverse_scale'])


## 4. Dispatch и stateful настройки

`_encode` проверяет `use_framewise_decoding`, несмотря на имя операции. `use_framewise_encoding` в этом body не участвует. Public `encode` при opt_tiling=True вызывает planner и `apply_tiling`; public `decode` также пересчитывает policy и игнорирует дополнительные kwargs. Поэтому ручные flags могут быть перезаписаны следующим public call.

В опытах 04–07 используем `_encode/_decode` с явной конфигурацией. Это диагностический private API, а не обещание стабильности библиотеки. Для приложения нужен adapter с версионированным контрактом, shape guards и проверкой postconditions.


In [ ]:
show_source('_encode');show_source('decode');show_source('apply_tiling')
metrics['dispatch_notes']='_encode uses use_framewise_decoding; public decode recomputes tile policy'


## Упражнения и критерий перехода

1. Нарисуйте tensor dimensions для released latent=16 и tiny latent=2; не смешивайте их.
2. Создайте adapter caller→posterior.mode→scale→layout и проверьте batch=2.
3. Какой дополнительный источник нужен, чтобы утверждать, что optional attention включена в конкретном checkpoint? Переход: объясните каждый тип значения на границе encode/DiT/decode.

Сначала запишите прогноз, затем меняйте один фактор. Разделяйте ошибку аппроксимации, ошибку интерфейса/формы и аппаратное ограничение. Сохранённые метрики относятся к этому запуску.


In [ ]:
result=save_metrics(LAB,metrics)
print(json.dumps(result,ensure_ascii=False,indent=2))


## Источники

[HunyuanVideo, v6, §4.1 / Fig. 6 / Eq. 1, стр. 5–6](https://arxiv.org/pdf/2412.03603v6) · [Auto-Encoding Variational Bayes](https://arxiv.org/abs/1312.6114) · [Официальный HunyuanVideo](https://github.com/Tencent-Hunyuan/HunyuanVideo) · [Закреплённый исследуемый исходник](../../kandinsky-6/reference/kandinsky-6/kandinsky/core/components/hunyuan_vae.py) · [Lecture 4 / исходный MIT Lab 3](../../lecture_04/README.md).

Математические выводы и малые опыты — авторские пояснения. Числа tiny-модели не являются оценкой pretrained HunyuanVideo/Kandinsky. Подробная provenance и ограничения — в SOURCE_MAP.
